# EDA - Marathon

In [0]:
VOLUME_PATH = '/Volumes/marathos/default/raw'
RAWDATA_PATH = VOLUME_PATH + '/data'
RAWDATA_FILE = RAWDATA_PATH + '/TWO_CENTURIES_OF_UM_RACES.csv'

auto_schema = (
    spark.read.format("csv")
    .options(header=True, inferSchema=True)
    .load(f"{RAWDATA_FILE}")
    .schema
)

auto_schema

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

real_schema = StructType([
    StructField('Year of event', IntegerType(), True),
    StructField('Event dates', StringType(), True),
    StructField('Event name', StringType(), True),
    StructField('Event distance/length', StringType(), True),
    StructField('Event number of finishers', IntegerType(),True),
    StructField('Athlete performance', StringType(), True),
    StructField('Athlete club', StringType(), True),
    StructField('Athlete country', StringType(), True),
    StructField('Athlete year of birth', DoubleType(), True),
    StructField('Athlete gender', StringType(), True),
    StructField('Athlete age category', StringType(), True),
    StructField('Athlete average speed', StringType(), True),
    StructField('Athlete ID', IntegerType(), True)])

df_marathon = (
    spark.read.format("csv")
    .options(header=True, schema=real_schema)
    .load(f"{RAWDATA_FILE}")
)

df_marathon.createOrReplaceTempView("df_marathon_view")

df_marathon.limit(10).display()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count the number of null values in each column
null_counts = df_marathon.select(
    [spark_sum(col(column).isNull().cast("int")).alias(column) for column in df_marathon.columns]
)

# Convert the result to a dictionary so that we can loop through it
null_counts = null_counts.collect()[0].asDict()

# Only keep columns with null values
[(column, nulls) for column, nulls in null_counts.items() if nulls > 0]

In [0]:
%sql
SELECT
    count(*) AS nr_rows
FROM df_marathon_view;

In [0]:
%sql
SELECT DISTINCT
    `Event distance/length` AS event_goal,
    CASE 
        WHEN `Event distance/length` LIKE "%km%" THEN "km"
        WHEN `Event distance/length` LIKE "%mi%" THEN "km"
        WHEN `Event distance/length` LIKE "%h%" THEN "h"
    END AS Unit
FROM df_marathon_view
WHERE `Event distance/length` LIKE "%h%"
LIMIT 40;

## Initial Analys

- Rows: 7 461 195
- Columns

| Column | Shows |Datatype now|Datatype should be|Nr nulls|
|--------|-------|------------|-------------------|--------|
|Year of event|A year (YYYY) for the event|Integer|Integer|0|
|Event dates|The date for the event|String|Date|0|
|Event name|word|String|String|0|
|Event distance/length|word|String|String|0|
|Event number of finishers|word|Integer|Integer|0|
|Athlete performance|word|String|String|2|
|Athlete club|word|String|String|2826373|
|Athlete country|word|String|String|3|
|Athlete year of birth|word|Double|Double|588161|
|Athlete gender|word|String|String|7|
|Athlete age category|word|String|String|584938|
|Athlete average speed|word|String|String|224|
|Athlete ID|word|Integer|Integer|0|